# W1 - WISE: All Regions Plotted

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
import os
import shutil
from astropy.io import fits # importing fits 
from astropy.utils.data import download_file # simple method of downloading FITS file 
from astropy.visualization import astropy_mpl_style # resource for organized plots
from astropy.visualization import ZScaleInterval, MinMaxInterval, AsinhStretch, LogStretch, ImageNormalize # quality image display/visualization
from astropy.wcs import WCS #coordinate conversion
from regions import Regions # import astropy region class


In [5]:
class DisplayData: 
    """" A class to open FITS file, display image data, 
         and plot a DS9.reg file onto the image data. """ 
    def __init__(self, filepath):
        """ Initialize through opening the data.""" 
        self.filepath = filepath
        from astropy.io import fits 
        with fits.open(filepath) as hdul: # opening file
            data = None
            header = None
            for hdu in hdul: #loops through each extension (HDU) in the list of HDUs to check if data has dimensions (thanks rav!)
                if hdu.data is not None and hdu.header.get("NAXIS",0) >= 2:
                    data = hdu.data 
                    header = hdu.header 
                    print(f"Found data in extension: {hdu.name or hdul.index(hdu)}")
                    break 
            
        if data is None:
            print("None")
       # hdu = fits.open(filepath)[1] #okay index 1 isn't always gong to work. you need to write an if statement to test 0


    def display_header(self, filepath):
        self.filepath = filepath
        from astropy.io import fits 
        with fits.open(filepath) as hdul:
            data = None
            header = None
            for hdu in hdul:
                if hdu.data is not None and hdu.header.get("NAXIS",0) >= 2:
                    data = hdu.data 
                    header = hdu.header 
                    print(f"Found data in extension: {hdu.name or hdul.index(hdu)}")
                    break 
            
        if data is None:
            print("None")

    def display_image(self, filepath, norm, vmin, vmax):
        self.filepath = filepath 
        self.norm = norm
        self.vmin = vmin
        self.vmax = vmax
        #brightness control
        #hdu = fits.open(filepath)[1]
        #data = hdu.data
        #header = hdu.header 
        from astropy.io import fits
        with fits.open(filepath) as hdul:
            data = None
            header = None
            for hdu in hdul:
                if hdu.data is not None and hdu.header.get("NAXIS",0) >= 2:
                    data = hdu.data 
                    header = hdu.header 
                    print(f"Found data in extension: {hdu.name or hdul.index(hdu)}")
                    break 
            
        if data is None:
            print("None")



        plt.figure(figsize=(8, 8))
        plt.imshow(data, cmap="gray", origin="lower", norm=self.norm, vmin=self.vmin, vmax=self.vmax) #figure out a way to add normalization parameter into this function
        plt.colorbar(label="Intensity", fraction=0.0146, aspect=20)
    

    
    def plot_region(self, filepath, disk_regionfile, ring_regionfile, bar_regionfile, center_regionfile, spiralarms_regionfile, norm, vmin, vmax, figname):
        self.filepath = filepath
        self.disk_regionfile = disk_regionfile
        self.ring_regionfile = ring_regionfile
        self.bar_regionfile = bar_regionfile
        self.center_regionfile = center_regionfile
        self.spiralarms_regionfile = spiralarms_regionfile
        self.norm = norm 
        self.vmin = vmin
        self.vmax= vmax 
        self.figname = figname
        from astropy.io import fits 
        from astropy.wcs import WCS
        with fits.open(filepath) as hdul:
            data = None
            header = None
            for hdu in hdul:
                if hdu.data is not None and hdu.header.get("NAXIS",0) >= 2:
                    data = hdu.data 
                    header = hdu.header 
                    print(f"Found data in extension: {hdu.name or hdul.index(hdu)}")
                    break 
            
        if data is None:
            print("None")
            
        wcs = WCS(header) #see https://astronomy.stackexchange.com/questions/51673/how-to-slice-wcs-in-a-fits-file
        # access region file via parsing & converting data from wcs to pixel. the 0 is for the FIRST region plotted
        # for i in regionfile??? what i want to do is for each region in the regionfile, plot each region if an index exists for that region
      
        disk_region = Regions.read(disk_regionfile)[0].to_pixel(wcs)
        ring_region1 = Regions.read(ring_regionfile)[0].to_pixel(wcs) # access region file via
        ring_region2 = Regions.read(ring_regionfile)[1].to_pixel(wcs) # the one corresponds to
        bar_region = Regions.read(bar_regionfile)[0].to_pixel(wcs)
        bar_region2 = Regions.read (bar_regionfile)[1].to_pixel(wcs)
        center_region = Regions.read(center_regionfile)[0].to_pixel(wcs)
        spiralarms_region1 = Regions.read(spiralarms_regionfile)[0].to_pixel(wcs)
        spiralarms_region2 = Regions.read(spiralarms_regionfile)[1].to_pixel(wcs)
        
       
        fig, ax = plt.subplots(subplot_kw=dict(projection=wcs))
        ax.imshow(data, cmap='gray', origin='lower', norm=self.norm, vmin=self.vmin, vmax=self.vmax) 
        
        #https://docs.astropy.org/en/stable/visualization/wcsaxes/index.html
        
        
        overlay = ax.get_coords_overlay('fk5')
        overlay[0].set_axislabel('Right Ascension (J2000)')
        overlay[1].set_axislabel('Declination (J2000)')
        
        disk_region.plot(ax=ax, color='red', lw=2.0)
        ring_region1.plot(ax=ax, color= 'blue', lw=2.0) 
        ring_region2.plot(ax=ax, color='blue', lw=2.0)
        bar_region.plot(ax=ax, color= 'green', lw=1.0)
        bar_region2.plot(ax=ax, color= 'green', lw=1.0)
        center_region.plot(ax=ax, color='orange', lw=1.0)
        spiralarms_region1.plot(ax=ax, color='pink', lw=2.0)
        spiralarms_region2.plot(ax=ax, color='pink', lw=2.0)
        
        plt.savefig(figname, bbox_inches='tight')
                        

In [6]:
%matplotlib widget
w1_file= r"C:\Users\Lamat\OneDrive - The Ohio State University\Lamat NGC 253 Environment Mask\Plotting Regions\ngc0253_w1_bgsub.fits"
w1_data = DisplayData(w1_file)
w1_disk_regionfile = r"C:\Users\Lamat\OneDrive - The Ohio State University\Lamat NGC 253 Environment Mask\DS9 NGC 253 Updated Region Files\Disk\disk_w1_regionfile_wise.reg"
w1_ring_regionfile = r"C:\Users\Lamat\OneDrive - The Ohio State University/Lamat NGC 253 Environment Mask/Plotting Regions/ring_w1_regionfile_wise.reg"
w1_center_regionfile = r"C:\Users\Lamat\OneDrive - The Ohio State University/Lamat NGC 253 Environment Mask/Plotting Regions/center_w1_regionfile_wise1.reg"
w1_bar_regionfile = r"C:\Users\Lamat\OneDrive - The Ohio State University/Lamat NGC 253 Environment Mask/Plotting Regions/bar_w1_regionfile_wise_twoboxes.reg"
w1_spiralarms_regionfile = r"C:\Users\Lamat\OneDrive - The Ohio State University/Lamat NGC 253 Environment Mask/Plotting Regions/spiralarms_w1_regionfile_wise.reg"
w1_data.plot_region(w1_file, w1_disk_regionfile, w1_ring_regionfile, w1_bar_regionfile, w1_center_regionfile, w1_spiralarms_regionfile, norm = 'log', vmin=0.05, vmax=255, figname='W1 map')

Found data in extension: PRIMARY
Found data in extension: PRIMARY


C:\Users\Lamat\AppData\Local\Programs\Python\Python314\Lib\site-packages\regions\shapes\ellipse.py:222: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  return Ellipse(xy=xy, width=width, height=height, angle=angle,
C:\Users\Lamat\AppData\Local\Programs\Python\Python314\Lib\site-packages\regions\shapes\rectangle.py:219: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  return Rectangle(xy=xy, width=width, height=height,
C:\Users\Lamat\AppData\Local\Programs\Python\Python314\Lib\site-packages\regions\shapes\polygon.py:225: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  return Polygon(xy=xy, **mpl_kwargs)
